Unlike the Black-Scholes model, which is a continuous-time formula, the binomial model breaks time into discrete steps, allowing us to value options that can be exercised early (American options).

The model assumes that in each time step, the stock price $S$ can either go up by a factor $u$ or down by a factor $d$.
- Up factor ($u$): $e^{\sigma \sqrt{\Delta t}}$
- Down factor ($d$): $1/u$
- Risk-neutral probability ($p$): $\frac{e^{(r - q) * \Delta t} - d }{u - d}$

The risk-neutral probability is derived from the following martingale assumption (note that $r$ gets modified by $(r - q)$):

$$S = e^{-(r-q)\Delta t} \times E[S_{\text{future}}].$$
This implies:
$$S = e^{-(r-q)\Delta t} [p(S\cdot u) + (1 - p)(S\cdot d)].$$

<p align="center">
  <img src="https://raw.githubusercontent.com/MilindGunjal/Trinomial-trees/master/binomtree.png" width="750">
</p>

Methodology:

- Step 1: Forward Induction (The Stock Price Tree)

  We start at $t=0$ with the current stock price and calculate all possible future prices until the expiration date $T$.

- Step 2: Backward Induction (The Option Price Tree)

  We start at the final nodes (expiration) where the option value is simply its intrinsic value:

  - Call: $\max(S_T - K, 0)$
  - Put: $\max(K - S_T, 0)$
  
  Then, we work backward to $t=0$ using the risk-neutral formula:
  $$V_t = e^{-r \Delta t} [p V_{u} + (1-p) V_{d}]$$

The key difference:

- European Options: You only calculate the discounted value from the future nodes.

- American Options: At every single node, you compare the discounted value vs. the immediate exercise value (using step 1 data). You take the higher of the two.

In [1]:
import numpy as np
from scipy.stats import norm

def binomial_tree_option(S, K, T, r, sigma, N, q=0.0, option_type='call', exercise_style='European'):
    """
    S: Current stock price
    K: Strike price
    T: Time to maturity (years)
    r: Risk-free rate
    sigma: Volatility
    N: Number of time steps
    q = Dividend yield
    option_type: 'call' or 'put'
    exercise_style: 'European' or 'American'
    """

    # 1. Calculate constants
    dt = T / N
    u = np.exp(sigma * np.sqrt(dt))
    d = 1 / u

    # RISK-NEUTRAL PROBABILITY
    p = (np.exp((r - q) * dt) - d) / (u - d)
    discount = np.exp(-r * dt)

    # 2. Initialize stock prices at maturity
    S_at_t = S * (u ** np.arange(N + 1)) * (d ** np.arange(N, -1, -1)) # Initializing Step 1 (from down most node to up most node)

    # 3. Initialize option values at maturity
    if option_type == 'call':
        V = np.maximum(S_at_t - K, 0)
    else:
        V = np.maximum(K - S_at_t, 0)

    # 4. Step backward through the tree
    for j in range(N - 1, -1, -1):
        S_at_t = S * (u ** np.arange(j + 1)) * (d ** np.arange(j, -1, -1)) # Executing Step 1
        V = discount * (p * V[1:] + (1 - p) * V[:-1]) # Executing Step 2

        # 5. American Exercise Logic
        if exercise_style == 'American':
            if option_type == 'call':
                V = np.maximum(V, S_at_t - K)
            else:
                V = np.maximum(V, K - S_at_t)

    return V[0]

def black_scholes_merton(S, K, T, r, sigma, q=0.0, option_type='call'): # Merton variation accomodates the dividend yield q
    """
    S: Current stock price
    K: Strike price
    T: Time to maturity (years)
    r: Risk-free rate
    sigma: Volatility
    q: Dividend yield
    option_type: 'call' or 'put'
    """

    # Calculate d1 and d2
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T)) # Modified to accomodate the dividend yield q
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == 'call':
        price = (S * np.exp(-q * T) * norm.cdf(d1)) - (K * np.exp(-r * T) * norm.cdf(d2)) # Modified to accomodate the dividend yield q
    elif option_type == "put":
        price = (K * np.exp(-r * T) * norm.cdf(-d2)) - (S * np.exp(-q * T) * norm.cdf(-d1)) # Modified to accomodate the dividend yield q

    return price

# --- Testing with an example ---
params = {
    "S": 100,      # Stock Price
    "K": 125,      # Strike Price
    "T": 1,        # 1 Year
    "r": 0.04,     # 4% Interest
    "sigma": 0.2,  # 20% Volatility
    "q": 0.08      # 8% Dividend yield
}

N = 1000 # steps

print(f"American Call: {binomial_tree_option(**params, N = N, option_type='call', exercise_style='American'):.4f}")
print(f"European Call: {binomial_tree_option(**params, N = N, option_type='call', exercise_style='European'):.4f}")
print(f"Black-Scholes-Merton Call Price: {black_scholes_merton(**params, option_type='call'):.4f}")
print(f"American Put: {binomial_tree_option(**params, N = N, option_type='put', exercise_style='American'):.4f}")
print(f"European Put: {binomial_tree_option(**params, N = N, option_type='put', exercise_style='European'):.4f}")
print(f"Black-Scholes-Merton Put Price: {black_scholes_merton(**params, option_type='put'):.4f}")


American Call: 0.9676
European Call: 0.9244
Black-Scholes-Merton Call Price: 0.9239
American Put: 28.7124
European Put: 28.7115
Black-Scholes-Merton Put Price: 28.7110


Analysing the results:

- As $N \longrightarrow \infty$, European option prices $\longrightarrow$ Black-Scholes-Merton option price.

- Ameican Vs. European options:
  - Case 1: $q > 0$
  
    American call as well as put are valued at a premium, compared to the European ones.
  
    Why?

    Exercising options before the expiry can be more profitable. In case of an early exercise, the "time value" of the money you'd get from exercising now might be worth more than waiting for the stock to potentially drop further.
  
  - Case 2: $q = 0$
  
    American put prices are at a premium compared to the European put, however, American calls are valued the same as European calls.

    Why?

    - The Time Value of Money
    
      The primary reason American Puts are exercised early, while Calls are not, comes down to how we treat the strike price $K$. For a Put: When you exercise, you receive $K$ (cash) immediately. Money today is worth more than money tomorrow because you can reinvest it at the risk-free rate.  If the stock is deep in-the-money, the interest you earn on that cash outweighs any potential further drop in the stock. For a Call: When you exercise, you pay $K$ (cash) immediately. Why pay cash now when you could keep it in a bank, earn interest, and wait until the very last second to hand it over?
    - Insurance and Upside
    
      There are two components to an option's value: Intrinsic Value (what it's worth right now) and Time Value (the "insurance" against volatility). Calls: If you exercise a call early, you capture the intrinsic value but you sacrifice the insurance. If the stock crashes to zero tomorrow, a call holder loses nothing (the option just expires worthless), but a stockholder (who exercised early) loses the entire investment. Why give up that protection for free? Puts: A put is insurance against a price drop. However, a stock price cannot go below zero. If a company goes bankrupt and the stock hits $0, the put has reached its maximum possible value. There is no more "insurance" left to provide, so you should exercise immediately to get your cash and start earning interest.

Doesn't the 'Put-Call Parity' break if the call prices remain the same but the put prices change?

- European options: Put-Call Parity holds only for European options:

$$C_{\text{Euro}} - P_{\text{Euro}} = S_0 e^{-qT} - K e^{-rT}$$

- American options: Equivalent of Put-Call Parity for American options are the following inequalities:

$$S_0 e^{-qT} - K \leq C_{\text{Amer}} - P_{\text{Amer}} \leq S_0 - Ke^{-rT}$$

Now we study Trinomial tree option pricing method. Trinomial Tree is often preferred in professional finance because it converges to the correct price much faster and provides a more stable model of stock price movements (as opposed to binomial, where the values might oscillate as we change $N$ from even to odd).

 In a trinomial model, we allow for three possibilities in each time step $\Delta t$:

 - Up: The stock price becomes $S\cdot u$
 - Middle (Stay): The stock price stays $S$ (or moves slightly to $S\cdot m$)
 - Down: The stock price becomes $S\cdot d$

<p align="center">
  <img src="https://raw.githubusercontent.com/MilindGunjal/Trinomial-trees/master/trinomtree.png" width="500">
</p>

 To maintain consistency with volatility ($\sigma$) and the risk-free rate ($r$), the standard (Boyle) factors are:

 - $u = e^{\sigma \sqrt{2 \Delta t}}$
 - $m = 1$ (The price stays the same)
 - $d = 1/u = e^{-\sigma \sqrt{2 \Delta t}}$

 Similar to the binomial tree, we have risk-neutral probabilities that can be derived from binomial tree calculations by breaking a $\Delta t$ step down to two $(\Delta t) / 2$ steps. One can visualize this by equating the results of following two trees.

<img src="https://raw.githubusercontent.com/MilindGunjal/Trinomial-trees/master/binomsmalltree.png" width="49%"> <img src="https://raw.githubusercontent.com/MilindGunjal/Trinomial-trees/master/trinomsmalltree.png" width="49%">

 In this case, $d$ becomes $d^{1/2}$ and $u$ becomes $u^{1/2}$. So, we get the probability:



 $$P = \frac{e^{(r-q)\Delta t / 2} - d^{1/2}}{u^{1/2} - d^{1/2}}$$

 Moreover,

 - $p_u = P^2$
 - $p_m = 2P(1 - P)$
 - $p_d = (1 - P)^2$

 So finally, we get the risk-neutral probabilities as follows:

 - $p_u = \left( \frac{e^{(r-q)\frac{\Delta t}{2}} - e^{-\sigma \sqrt{\frac{\Delta t}{2}}}}{e^{\sigma \sqrt{\frac{\Delta t}{2}}} - e^{-\sigma \sqrt{\frac{\Delta t}{2}}}} \right)^2$
 - $p_d = \left( \frac{e^{\sigma \sqrt{\frac{\Delta t}{2}}} - e^{(r-q)\frac{\Delta t}{2}}}{e^{\sigma \sqrt{\frac{\Delta t}{2}}} - e^{-\sigma \sqrt{\frac{\Delta t}{2}}}} \right)^2$
 - $p_m = 1 - (p_u + p_d)$


In [2]:
import numpy as np
from scipy.stats import norm

def binomial_tree_option(S, K, T, r, sigma, N, q=0.0, option_type='call', exercise_style='European'):
    """
    S: Current stock price
    K: Strike price
    T: Time to maturity (years)
    r: Risk-free rate
    sigma: Volatility
    N: Number of time steps
    q = Dividend yield
    option_type: 'call' or 'put'
    exercise_style: 'European' or 'American'
    """

    # 1. Calculate constants
    dt = T / N
    u = np.exp(sigma * np.sqrt(dt))
    d = 1 / u

    # RISK-NEUTRAL PROBABILITY
    p = (np.exp((r - q) * dt) - d) / (u - d)
    discount = np.exp(-r * dt)

    # 2. Initialize stock prices at maturity
    S_at_t = S * (u ** np.arange(N + 1)) * (d ** np.arange(N, -1, -1)) # Initializing Step 1 (from down most node to up most node)

    # 3. Initialize option values at maturity
    if option_type == 'call':
        V = np.maximum(S_at_t - K, 0)
    else:
        V = np.maximum(K - S_at_t, 0)

    # 4. Step backward through the tree
    for j in range(N - 1, -1, -1):
        S_at_t = S * (u ** np.arange(j + 1)) * (d ** np.arange(j, -1, -1)) # Executing Step 1
        V = discount * (p * V[1:] + (1 - p) * V[:-1]) # Executing Step 2

        # 5. American Exercise Logic
        if exercise_style == 'American':
            if option_type == 'call':
                V = np.maximum(V, S_at_t - K)
            else:
                V = np.maximum(V, K - S_at_t)

    return V[0]


def trinomial_tree_option(S, K, T, r, sigma, N, q=0.0, option_type='call', exercise_style='European'):
    # 1. Constants
    dt = T / N
    u = np.exp(sigma * np.sqrt(2 * dt))
    d = 1/u
    m = 1

    # 2. Risk-neutral probabilities
    # Simplified version for implementation
    p_u = ((np.exp((r - q) * dt / 2) - np.exp(-sigma * np.sqrt(dt / 2))) /
           (np.exp(sigma * np.sqrt(dt / 2)) - np.exp(-sigma * np.sqrt(dt / 2))))**2
    p_d = ((np.exp(sigma * np.sqrt(dt / 2)) - np.exp((r - q) * dt / 2)) /
           (np.exp(sigma * np.sqrt(dt / 2)) - np.exp(-sigma * np.sqrt(dt / 2))))**2
    p_m = 1 - (p_u + p_d)

    discount = np.exp(-r * dt)

    # 3. Initialize stock prices at maturity
    # At step N, there are 2N + 1 nodes
    # Prices range from S*u^N down to S*d^N
    S_at_t = S * (u ** np.arange(N, -N - 1, -1))

    # 4. Initialize option values at maturity
    if option_type == 'call':
        V = np.maximum(S_at_t - K, 0)
    else:
        V = np.maximum(K - S_at_t, 0)

    # 5. Backward induction
    for j in range(N - 1, -1, -1):
        # Discounted expected value using 3 nodes (u, m, d)
        V = discount * (p_u * V[:-2] + p_m * V[1:-1] + p_d * V[2:])

        # American exercise check
        if exercise_style == 'American':
            # Current stock prices for this level
            S_at_t = S * (u ** np.arange(j, -j - 1, -1))
            if option_type == 'call':
                V = np.maximum(V, S_at_t - K)
            else:
                V = np.maximum(V, K - S_at_t)

    return V[0]

def black_scholes_merton(S, K, T, r, sigma, q=0.0, option_type='call'): # Merton variation accomodates the dividend yield q
    """
    S: Current stock price
    K: Strike price
    T: Time to maturity (years)
    r: Risk-free rate
    sigma: Volatility
    q: Dividend yield
    option_type: 'call' or 'put'
    """

    # Calculate d1 and d2
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T)) # Modified to accomodate the dividend yield q
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == 'call':
        price = (S * np.exp(-q * T) * norm.cdf(d1)) - (K * np.exp(-r * T) * norm.cdf(d2)) # Modified to accomodate the dividend yield q
    elif option_type == "put":
        price = (K * np.exp(-r * T) * norm.cdf(-d2)) - (S * np.exp(-q * T) * norm.cdf(-d1)) # Modified to accomodate the dividend yield q

    return price

# --- Testing with an example ---
params = {
    "S": 100,      # Stock Price
    "K": 125,      # Strike Price
    "T": 1,        # 1 Year
    "r": 0.04,     # 4% Interest
    "sigma": 0.2,  # 20% Volatility
    "q": 0.08      # 8% Dividend yield
}

N = 1000 # steps

print(f"Binomial American Call Price: {binomial_tree_option(**params, N = N, option_type='call', exercise_style='American'):.4f}")
print(f"Trinomial American Call Price: {trinomial_tree_option(**params, N = N, option_type='call', exercise_style='American'):.4f}")
print('\n')
print(f"Binomial European Call Price: {binomial_tree_option(**params, N = N, option_type='call', exercise_style='European'):.4f}")
print(f"Trinomial European Call Price: {trinomial_tree_option(**params, N = N, option_type='call', exercise_style='European'):.4f}")
print(f"Black-Scholes-Merton Call Price: {black_scholes_merton(**params, option_type='call'):.4f}")
print('\n')
print(f"Binomial American Put Price: {binomial_tree_option(**params, N = N, option_type='put', exercise_style='American'):.4f}")
print(f"Trinomial American Put Price: {trinomial_tree_option(**params, N = N, option_type='put', exercise_style='American'):.4f}")
print('\n')
print(f"Binomial European Put Price: {binomial_tree_option(**params, N = N, option_type='put', exercise_style='European'):.4f}")
print(f"Trinomial European Put Price: {trinomial_tree_option(**params, N = N, option_type='put', exercise_style='European'):.4f}")
print(f"Black-Scholes-Merton Put Price: {black_scholes_merton(**params, option_type='put'):.4f}")


Binomial American Call Price: 0.9676
Trinomial American Call Price: 0.9668


Binomial European Call Price: 0.9244
Trinomial European Call Price: 0.9235
Black-Scholes-Merton Call Price: 0.9239


Binomial American Put Price: 28.7124
Trinomial American Put Price: 28.7116


Binomial European Put Price: 28.7115
Trinomial European Put Price: 28.7106
Black-Scholes-Merton Put Price: 28.7110


Binomial vs Trinomial:

- Both converge to the same answer for larger $N$ (i.e., trinomial converges to Black-Scholes-Merton in the European case).
- Speed (Efficiency): A trinomial tree with $N$ steps is roughly equivalent in accuracy to a binomial tree with $N^2$ steps.
- Stability: Binomial trees can oscillate (the price jumps up and down significantly between odd and even $N$ steps). The trinomial model provides a smoother path to the final price.
- Use Case: Binomial model is used for teaching & basic American options. The trinomial model is industry standard for complex American/Exotic options.

In financial engineering, the "Greeks" measure the sensitivity of an option's price to various market parameters. When using tree-based models (Binomial or Trinomial), we calculate these by looking at the differences in option values across neighboring nodes at the start of the tree.

1. Delta ($\Delta$): The Hedge Ratio
  
Delta measures how much the option price ($V$) changes for a $\$1$ change in the underlying stock price ($S$). It also represents the number of shares you should hold to "hedge" the option.

- $\Delta_{\text{Call}} \in [0,1], \Delta_{\text{Put}} \in [-1, 0]$

- Trinomial Calculation: We look at the first step ($t = 1$). We have three prices: $V_u$ (up), $V_m$ (middle), and $V_d$ (down), corresponding to stock prices $S_u, S_m,$ and $S_d$.$$\Delta = \frac{V_u - V_d}{S_u - S_d}$$

  As the formula suggests, we can also use binomial trees to calculate $\Delta$.

2. Gamma ($\Gamma$): The Acceleration

Gamma measures the rate of change in Delta with respect to change in underlying stock price ($S$). It tells you how often you need to adjust your hedge. If Gamma is high, your Delta changes rapidly as the stock moves.

- Trinomial Calculation: $$\Gamma = \frac{\left( \frac{V_u - V_m}{S_u - S_m} \right) - \left( \frac{V_m - V_d}{S_m - S_d} \right)}{0.5(S_u - S_d)}.$$

  As the formula suggests, we cannot use binomial trees to calculate $\Gamma$ directly. This is a novelty for trinomial trees.

3. Theta ($\Theta$): Time Decay

Theta measures how much value the option loses every day as it approaches expiration, assuming the stock price doesn't move. Theta is usually negative.

- Trinomial Calculation: We compare the option value at $t=0$ ($V_0$) with the value at the middle node at $t=1$ ($V_m$), because $V_m$ represents the price if the stock stayed exactly the same after one time step $\Delta t$.$$\Theta = \frac{V_m - V_0}{\Delta t}.$$

  As the formula suggests, we cannot use binomial trees to calculate $\Theta$ directly. This is a novelty for trinomial trees.

In [3]:
import numpy as np

def trinomial_greeks(S, K, T, r, sigma, N, q=0.0, option_type='call', exercise_style='European'):
    dt = T / N
    u = np.exp(sigma * np.sqrt(2 * dt))
    d = 1/u

    # Probabilities
    p_u = ((np.exp((r - q) * dt / 2) - np.exp(-sigma * np.sqrt(dt / 2))) /
           (np.exp(sigma * np.sqrt(dt / 2)) - np.exp(-sigma * np.sqrt(dt / 2))))**2
    p_d = ((np.exp(sigma * np.sqrt(dt / 2)) - np.exp((r - q) * dt / 2)) /
           (np.exp(sigma * np.sqrt(dt / 2)) - np.exp(-sigma * np.sqrt(dt / 2))))**2
    p_m = 1 - (p_u + p_d)
    discount = np.exp(-r * dt)

    # 1. Initialize stock prices at maturity (Step N)
    S_at_t = S * (u ** np.arange(N, -N - 1, -1))

    # 2. Option values at maturity
    if option_type == 'call':
        V = np.maximum(S_at_t - K, 0)
    else:
        V = np.maximum(K - S_at_t, 0)

    # 3. Backward induction
    for j in range(N - 1, -1, -1):
        V = discount * (p_u * V[:-2] + p_m * V[1:-1] + p_d * V[2:])

        # American exercise check
        if exercise_style == 'American':
            S_current = S * (u ** np.arange(j, -j - 1, -1))
            V = np.maximum(V, S_current - K if option_type == 'call' else K - S_current)

        # Capture values at step j=1 to calculate Greeks
        if j == 1:
            V_u, V_m, V_d = V[0], V[1], V[2]
            S_u, S_m, S_d = S*u, S, S*d

    # 4. Final Option Price at j=0
    V_0 = V[0]

    # 5. Greek Calculations
    delta = (V_u - V_d) / (S_u - S_d)
    gamma = ((V_u - V_m)/(S_u - S_m) - (V_m - V_d)/(S_m - S_d)) / (0.5 * (S_u - S_d))
    theta = (V_m - V_0) / dt # Value per year; divide by 365 for daily, since dt is valued in years

    return {
        "Option Price": V_0,
        "Delta": delta,
        "Gamma": gamma,
        "Theta (Annual)": theta,
        "Theta (Daily)": theta / 365
    }

# --- Testing with an example ---

params = {
    "S": 100,      # Stock Price
    "K": 125,      # Strike Price
    "T": 1,        # 1 Year
    "r": 0.04,     # 4% Interest
    "sigma": 0.2,  # 20% Volatility
    "q": 0.08      # 8% Dividend yield
}

N = 1000 # steps

results = trinomial_greeks(**params, N = N, option_type='call', exercise_style = 'American')
for greek, val in results.items():
    print(f"{greek}: {val:.5f}")

Option Price: 0.96679
Delta: 0.10932
Gamma: 0.00947
Theta (Annual): -1.41723
Theta (Daily): -0.00388


Now to calculate Vega ($\nu$) and Rho ($\rho$), instead of looking at the tree nodes, we treat the entire option pricing function as a "black box". We change one input by a tiny amount ($\epsilon$), recalculate the price, and see how much it changed. Mathematically, we use the central difference for better accuracy.
$$\text{Greek} \approx \frac{V(\text{input} + \epsilon) - V(\text{input} - \epsilon)}{2\epsilon}$$
- Vega ($\nu$): Sensitivity to Volatility (input = $\sigma$).
- Rho ($\rho$): Sensitivity to the Risk-free rate (input = $r$).

In [4]:
def trinomial_more_greeks(S, K, T, r, sigma, N, q=0.0, option_type='call', exercise_style='European'):
    # 1. Base Price
    base_price = trinomial_tree_option(S, K, T, r, sigma, N, q, option_type, exercise_style)

    # 2. Delta & Gamma (From internal nodes as shown previously)

    # 3. Vega Calculation
    sigma_bump = 0.0001
    price_sigma_up = trinomial_tree_option(S, K, T, r, sigma + sigma_bump, N, q, option_type, exercise_style)
    price_sigma_down = trinomial_tree_option(S, K, T, r, sigma - sigma_bump, N, q, option_type, exercise_style)
    vega = (price_sigma_up - price_sigma_down) / (2*sigma_bump)

    # 4. Rho Calculation
    r_bump = 0.0001
    price_r_up = trinomial_tree_option(S, K, T, r + r_bump, sigma, N, q, option_type, exercise_style)
    price_r_down = trinomial_tree_option(S, K, T, r - r_bump, sigma, N, q, option_type, exercise_style)
    rho = (price_r_up - price_r_down) / (2*r_bump)

    return {
        "Option Price": base_price,
        "Vega (per 1% vol)": vega / 100, # Often expressed as change per 1% move
        "Rho (per 1% rate)": rho / 100   # Often expressed as change per 1% move
    }

# --- Testing with an example ---

params = {
    "S": 100,      # Stock Price
    "K": 125,      # Strike Price
    "T": 1,        # 1 Year
    "r": 0.04,     # 4% Interest
    "sigma": 0.2,  # 20% Volatility
    "q": 0.08      # 8% Dividend yield
}

N = 1000 # steps

greeks = trinomial_more_greeks(**params, N = N, option_type='call', exercise_style = 'American')
print(f"Option Price: {greeks['Option Price']:.5f}")
print(f"Vega:  {greeks['Vega (per 1% vol)']:.5f}")
print(f"Rho:   {greeks['Rho (per 1% rate)']:.5f}")

Option Price: 0.96679
Vega:  0.18617
Rho:   0.08741
